Author: Sabrina Derwent

In [26]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

In [ ]:
#data_path = "C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Spring 2025\\STAT390\\LegalAid\\Data\\CAR_-_EP_Flow_Activity_Queue__Agent_Names\\"

In [27]:
data_path = "/Users/sabrinaderwent/Desktop/fall25/stat390/CAR_datasets/"

### User input ends

### Reading all filenames in the data folder

In [28]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [29]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

1 CAR - EP, Flow, Activity, Queue, & Agent Names (01-12-25 - 01-18-25) (52010, 7)
2 CAR - EP, Flow, Activity, Queue, & Agent Names (01-19-25 - 02-01-25) (95495, 7)
3 CAR - EP, Flow, Activity, Queue, & Agent Names (02-02-25 - 02-15-25) (90056, 7)
4 CAR - EP, Flow, Activity, Queue, & Agent Names (02-16-25 - 03-01-25) (88186, 7)
5 CAR - EP, Flow, Activity, Queue, & Agent Names (03-02-25 - 03-15-25) (86377, 7)
6 CAR - EP, Flow, Activity, Queue, & Agent Names (04-07-24 - 04-20-24) (88766, 7)
7 CAR - EP, Flow, Activity, Queue, & Agent Names (04-21-24 - 05-04-24) (89643, 7)
8 CAR - EP, Flow, Activity, Queue, & Agent Names (05-05-24 - 05-18-24) (82575, 7)
9 CAR - EP, Flow, Activity, Queue, & Agent Names (05-19-24 - 06-01-24) (71103, 7)
10 CAR - EP, Flow, Activity, Queue, & Agent Names (06-02-24 - 06-15-24) (84354, 7)
11 CAR - EP, Flow, Activity, Queue, & Agent Names (06-16-24 - 06-29-24) (82124, 7)
12 CAR - EP, Flow, Activity, Queue, & Agent Names (06-30-24 - 07-13-24) (79752, 7)
13 CAR - EP, 

In [30]:
df_main.head()


,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason
0,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,NaN,2025/01/14 12:39:32 PM,NaN,NaN,NaN
1,001a3748-8d50-4550-8461-33547983deb0,NaN,LACMain,NaN,2025/01/14 12:39:32 PM,NaN,NaN,NaN
2,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025/01/14 12:39:32 PM,NaN,NaN,NaN
3,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,LACMain,NaN,2025/01/14 12:39:32 PM,NaN,NaN,NaN
4,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,MainMenu,2025/01/14 12:39:45 PM,NaN,NaN,NaN


### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [31]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [32]:
# Checking the datatype of all columns
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [33]:
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [34]:
df_main["date"] = df_main["Activity Start Timestamp"].dt.date
df_main["date"] = pd.to_datetime(df_main["date"])

#### Exploring Menu Options in Call Journey

In [112]:
df_main["Termination Reason"].unique()
#df_main["EP Name"].unique()
#df_main["Activity Name"].unique()
#df_main["Flow Name"].unique()


array([nan, 'Customer Left', 'Queue Timeout', 'CUSTOMER_UNAVAILABLE',
       'System disconnected the contact', 'Agent Left',
       'RONA_TIMER_EXPIRED', 'RONA Timer Expired',
       'MAX_CALLBACK_RETRY_LIMIT_REACHED', 'NO_ANSWER_FROM_CUSTOMER',
       'NO_ANSWER_FROM_AGENT', 'System Error', 'AGENT_ENDS',
       'CUSTOMER_BUSY', 'NO_ANSWER_USER', 'NO_ANSWER_CUSTOMER',
       'USER_UNAVAILABLE', 'Participant Invite timer expired',
       'CONTACT_CALLBACK_IN_PROGRESS', 'OUTDIAL_FAILED', 'USER_BUSY',
       'AGENT_UNAVAILABLE', 'MEDIA_MANAGER_INTERNAL_ERROR',
       'CHANNEL_FAILURE', 'USER_DECLINED', 'AGENT_BUSY'], dtype=object)

For presentation 1, I spent a lot of time tyring to conceptualize what a call journey looks like, as the dataset can sometimes be ambiguous. For the next presentation, my goal is to develop a systematic way to trace a call journey despite some confusing journeys

### Filtering Dataset

In [36]:
# filtering dataset so that I only have datapoints that contain the "Termination Reason" column
start_date = "2025-03-16"
end_date = "2025-09-30"
df_short = df_main[(df_main['date'] >= start_date) & (df_main["date"] <= end_date)]

## Analyzing Termination Reason

In [ ]:
# looking at how many calls have a what termination reason
df_short.groupby( "Termination Reason").nunique()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,hour,date
Termination Reason,,,,,,,,,
AGENT_BUSY,12,2,0,0,38,8,4,11,12
AGENT_ENDS,73,2,0,0,196,13,10,11,54
AGENT_UNAVAILABLE,19,2,0,0,19,5,1,7,6
Agent Left,13228,13,0,0,13109,57,12,24,172
CHANNEL_FAILURE,1,1,0,0,1,1,1,1,1
CONTACT_CALLBACK_IN_PROGRESS,2,0,0,0,2,2,0,2,2
CUSTOMER_BUSY,28,2,0,0,79,5,8,12,23
CUSTOMER_UNAVAILABLE,65,2,0,0,192,14,11,14,47
Customer Left,75799,16,13,0,80443,58,12,24,199


In [ ]:
# creating a long dataframe that lists each step in the same column

df_long = (
    df.melt(
        id_vars=['Contact Session ID', "Activity Start Timestamp"],
        value_vars = ['EP Name', 'Flow Name', 'Activity Name', 'Queue Name', 'Termination Reason'],
        var_name='Step Category',
        value_name='Activity'
    )
    .dropna(subset=['Activity'])
    .reset_index()
)


In [ ]:
# making sure dataframe is in chronological order
df_long = df_long.sort_values(by=["Contact Session ID", "Activity Start Timestamp"])
df_long.head(100)

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
32121,42940,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:42 AM,EP Name,Main Number Telephony EP
32122,42942,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:42 AM,EP Name,Main Number Telephony EP
47958,90919,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:42 AM,Flow Name,LACMain
68802,138898,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:42 AM,Activity Name,LanguageSelectionMenu
32123,42943,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:43 AM,EP Name,Main Number Telephony EP
...,...,...,...,...,...
67426,135890,001c0d31-8051-4020-9740-a85443bfe1e7,2025/03/27 04:25:04 PM,Activity Name,ClosedQueueMenu
80488,231847,001c0d31-8051-4020-9740-a85443bfe1e7,2025/03/27 04:25:04 PM,Termination Reason,Customer Left
13148,17650,003b7023-b7ae-46dc-9bc9-3b3eb810257c,2025/03/25 12:45:31 PM,EP Name,Main Number Telephony EP
13149,17652,003b7023-b7ae-46dc-9bc9-3b3eb810257c,2025/03/25 12:45:31 PM,EP Name,Main Number Telephony EP


### Looking at Number of Calls with 'Customer Left"
This termination reason is focused on more in depth by EDA team 1, so this I didn't do analysis for this reason

In [130]:
# get the last row per session
last_steps = df_long.groupby('Contact Session ID').tail(1)

# filter by desired end condition
ended_hangup = last_steps[last_steps['Activity'] == 'Customer Left']

# get the full journey for those calls
filtered_journeys = df_long[df_long['Contact Session ID'].isin(ended_hangup['Contact Session ID'])]

filtered_journeys.groupby("Contact Session ID").nth(3)

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
68802,138898,00031192-99dd-4eec-815e-1d54abedc153,2025/03/28 10:23:42 AM,Activity Name,LanguageSelectionMenu
44772,79422,0007108a-5d96-44c6-b0b0-70b248af81b5,2025/03/26 04:26:09 PM,Flow Name,LACMain
47116,87877,001c0d31-8051-4020-9740-a85443bfe1e7,2025/03/27 04:22:22 PM,Flow Name,LACMain
40879,65629,003b7023-b7ae-46dc-9bc9-3b3eb810257c,2025/03/25 12:45:31 PM,Flow Name,LACMain
37793,54626,00b27d9c-4c4b-4001-8434-9de1ed2043a1,2025/03/24 12:26:53 PM,Flow Name,LACMain
...,...,...,...,...,...
48800,93966,ffc64497-8d28-438e-8b40-b15e2cba4680,2025/03/28 02:33:55 PM,Flow Name,LACMain
68560,138393,ffc79b54-6cad-41c1-ab3b-fc497553cbac,2025/03/28 09:46:48 AM,Activity Name,LanguageSelectionMenu
46753,86599,ffca8984-8b5e-419a-8f2b-16b6635e643b,2025/03/27 02:48:50 PM,Flow Name,LACMain
48925,94418,ffd29027-0357-4d8c-8a1e-55e1bd44564a,2025/03/28 03:05:30 PM,Flow Name,LACMain


### Looking at Calls with 'Queue Timeout'

In [ ]:
# filter by desired end condition
queue_time = df_long[df_long['Activity'] == 'Queue Timeout']

# get the full journey for those calls
filtered_journeys = df_long[df_long['Contact Session ID'].isin(queue_time['Contact Session ID'])]

filtered_journeys.groupby("Contact Session ID").tail()




,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
1061,1453,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:34:21 AM,EP Name,Courtesy Callback Telephony EP
71090,145387,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:34:21 AM,Queue Name,Family
71091,145388,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:34:21 AM,Queue Name,Family
77288,193365,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:34:21 AM,Termination Reason,Agent Left
71100,145435,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:36:21 AM,Queue Name,Family
...,...,...,...,...,...
12610,16924,fa4b154d-1215-4019-8a11-538f6ac994c0,2025/03/25 12:03:35 PM,EP Name,Courtesy Callback Telephony EP
73022,160858,fa4b154d-1215-4019-8a11-538f6ac994c0,2025/03/25 12:03:35 PM,Queue Name,SubSenior Tenant
73023,160859,fa4b154d-1215-4019-8a11-538f6ac994c0,2025/03/25 12:03:35 PM,Queue Name,SubSenior Tenant
78493,208836,fa4b154d-1215-4019-8a11-538f6ac994c0,2025/03/25 12:03:35 PM,Termination Reason,Agent Left


In [ ]:
# journey of this call - I chose this call because it demonstrates a Queue Timeout and multiple courtesy callbacks
filtered_journeys[filtered_journeys['Contact Session ID'] == "0d5dec0a-ec28-4bb0-a728-5f848b543d8d"]

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
70933,143963,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:35 AM,Queue Name,Family
70934,143964,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:35 AM,Queue Name,Family
77159,191941,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:35 AM,Termination Reason,Customer Left
77160,191942,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:35 AM,Termination Reason,Queue Timeout
12,31,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,EP Name,Courtesy Callback Telephony EP
13,34,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,EP Name,Courtesy Callback Telephony EP
49394,95987,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,Activity Name,CallbackRetry
70935,143966,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,Queue Name,Family
70936,143967,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,Queue Name,Family
70937,143968,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,Queue Name,Family


#### Looking at Courtesy Calls after the call originally reaches 'Queue Timeout'

In [ ]:
# example of how to manually get the number of courtesy callbacks  
filtered_journeys[(filtered_journeys['Contact Session ID'] == "0d5dec0a-ec28-4bb0-a728-5f848b543d8d")
                    & (filtered_journeys['Activity'] == "Courtesy Callback Telephony EP")]

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
12,31,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,EP Name,Courtesy Callback Telephony EP
13,34,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/23 05:25:45 AM,EP Name,Courtesy Callback Telephony EP
231,330,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 02:25:55 AM,EP Name,Courtesy Callback Telephony EP
232,333,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 02:25:55 AM,EP Name,Courtesy Callback Telephony EP
1029,1410,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:33:14 AM,EP Name,Courtesy Callback Telephony EP
1030,1411,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:33:15 AM,EP Name,Courtesy Callback Telephony EP
1051,1439,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:33:55 AM,EP Name,Courtesy Callback Telephony EP
1061,1453,0d5dec0a-ec28-4bb0-a728-5f848b543d8d,2025/03/24 08:34:21 AM,EP Name,Courtesy Callback Telephony EP


In [208]:
# number of cortesy callbacks

callback = df_long[df_long['Activity'] == 'Courtesy Callback Telephony EP']

# get the full journey for those calls
filtered_journeys = df_long[df_long['Contact Session ID'].isin(callback['Contact Session ID'])]

filtered_journeys.groupby("Contact Session ID").nunique()

,index,Activity Start Timestamp,Step Category,Activity
Contact Session ID,,,,
043288e1-7c8a-47f4-9636-890cb16b6f00,87,32,5,32
050126b1-adb4-4010-b2f6-abf85e2c2907,71,23,5,33
08a8ec34-0b9f-4367-bdec-4b0a1fc37ebc,59,21,5,32
08e25efb-e662-4319-acba-4a34468b68c7,57,17,5,30
09bde664-d348-451a-bba5-b27c4448af73,52,18,5,23
...,...,...,...,...
f32aa7d9-eb45-4cba-8134-058c42e3e6d3,82,27,5,37
f898c3d7-2f43-4d19-a4a9-005f376daa40,69,25,5,35
fa4b154d-1215-4019-8a11-538f6ac994c0,82,24,5,33


### Looking at calls with 'Agent Left'
I determined that this termination reason can solely be classified as positive.

In [165]:
last_steps = df_long.groupby('Contact Session ID').tail(1)

# filter by desired end condition
ended_agent = last_steps[last_steps['Activity'] == 'Agent Left']

# get the full journey for those calls
filtered_journeys = df_long[df_long['Contact Session ID'].isin(ended_agent['Contact Session ID'])]

filtered_journeys.groupby("Contact Session ID").tail(2)

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
74605,173399,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:09:32 PM,Queue Name,Staff Directory English Transfer
79592,221377,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:09:32 PM,Termination Reason,Agent Left
75403,179041,0199a46c-ae24-4a88-9238-21bcf9afe2de,2025/03/27 10:36:42 AM,Queue Name,Staff Directory English Transfer
80091,227019,0199a46c-ae24-4a88-9238-21bcf9afe2de,2025/03/27 10:36:42 AM,Termination Reason,Agent Left
75952,183019,03921112-2c72-438a-923d-a2e5ec04ba48,2025/03/27 03:14:50 PM,Queue Name,Staff Directory English Transfer
...,...,...,...,...,...
78901,213148,fbaea9ad-cc57-4365-83f8-1daf90790b7a,2025/03/25 04:58:03 PM,Termination Reason,Agent Left
72105,153806,ff6b7a11-2a8a-49df-b925-34e4ce58ad12,2025/03/24 03:30:30 PM,Queue Name,Front Desk Transfer
77891,201784,ff6b7a11-2a8a-49df-b925-34e4ce58ad12,2025/03/24 03:30:30 PM,Termination Reason,Agent Left
75169,177363,ffef5316-dfc9-4d55-ba63-65461530a4ed,2025/03/27 08:53:35 AM,Queue Name,Criminal Records Voicemail Transfer


In [ ]:
filtered_journeys[filtered_journeys['Contact Session ID'] == "008d1660-7605-44ca-a9b3-11b8f81e61f3"]

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
21853,29287,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:20 PM,EP Name,Main Number Telephony EP
21854,29289,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:20 PM,EP Name,Main Number Telephony EP
44182,77266,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:20 PM,Flow Name,LACMain
62727,125245,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:20 PM,Activity Name,LanguageSelectionMenu
21855,29290,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:21 PM,EP Name,Main Number Telephony EP
44183,77268,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:21 PM,Flow Name,LACMain
21862,29298,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:37 PM,EP Name,Main Number Telephony EP
62732,125254,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:37 PM,Activity Name,MainMenu
21864,29300,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:54 PM,EP Name,Main Number Telephony EP
21865,29302,008d1660-7605-44ca-a9b3-11b8f81e61f3,2025/03/26 02:01:54 PM,EP Name,Main Number Telephony EP


### Looking at Calls with 'System disconnected the contact'
I determined that these calls can be solely classified as neutral.

In [218]:
last_steps = df_long.groupby('Contact Session ID').tail(1)

# filter by desired end condition
system_dis = last_steps[last_steps['Activity'] == 'System disconnected the contact']

# get the full journey for those calls
filtered_journeys = df_long[df_long['Contact Session ID'].isin(system_dis['Contact Session ID'])]

filtered_journeys.groupby("Contact Session ID").nth(2)

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
35581,47538,00b57171-1552-4187-9989-4e5a15343d70,2025/03/28 05:30:15 PM,EP Name,Main Number Telephony EP
6622,8894,03c9f177-59b3-4d4b-be06-cbde1e02217c,2025/03/24 02:22:19 PM,EP Name,Pre-Legal Menu Seniors Menu Telephony EP
39897,62082,0537a69f-4b04-4892-95e1-72c302e76961,2025/03/25 09:54:08 AM,Flow Name,LACMain
41991,69633,06c33df8-8216-4001-b399-9fcaad7e9d9b,2025/03/26 07:09:57 AM,Flow Name,LACMain
15952,21368,06ebcbce-3c5d-4d3f-a7b9-59c701b3e008,2025/03/25 05:59:55 PM,EP Name,Main Number Telephony EP
...,...,...,...,...,...
23773,31835,f7be87f1-7d62-4ec2-a04a-9ccb181a410e,2025/03/26 05:24:37 PM,EP Name,Main Number Telephony EP
35473,47399,f8ba2232-2954-4e8c-9b78-a2e19c284c94,2025/03/28 04:37:30 PM,EP Name,Main Number Telephony EP
8597,11529,fb618dd9-0508-4914-9bf4-046e4652525e,2025/03/25 07:59:45 AM,EP Name,Main Number Telephony EP
17268,23162,fc15bf4f-c0ca-49e7-8218-c98bf3f8aa45,2025/03/26 08:30:34 AM,EP Name,Main Number Telephony EP


In [ ]:
filtered_journeys[filtered_journeys['Contact Session ID']== "fdc56f2f-75b9-4642-8fcb-f05b7ba668b8"]

,index,Contact Session ID,Activity Start Timestamp,Step Category,Activity
23787,31852,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,EP Name,Main Number Telephony EP
23788,31854,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,EP Name,Main Number Telephony EP
23789,31855,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,EP Name,Main Number Telephony EP
44886,79831,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,Flow Name,LACMain
44887,79833,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,Flow Name,LACMain
63797,127810,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:10 PM,Activity Name,LanguageSelectionMenu
23790,31856,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:28 PM,EP Name,Main Number Telephony EP
63798,127812,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:28 PM,Activity Name,LanguageSelectionMenu
23791,31858,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:41 PM,EP Name,Closed Hours-Holidays Menu Telephony EP
44888,79835,fdc56f2f-75b9-4642-8fcb-f05b7ba668b8,2025/03/26 05:37:41 PM,Flow Name,ClosedHoursHolidaysMenu
